# Demo 05 : The Store

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · the-store

We port ARI510 demo18's namespace and key operations. The same namespace and key replace a profile document; a collection uses separate keys.

In [2]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
namespace = ("users", "u1", "profile")
store.put(namespace, "depot", {"text": "depot: Noord"})
store.put(namespace, "language", {"text": "language: Dutch"})
store.put(("users", "u2", "profile"), "depot", {"text": "depot: Zuid"})
print("profile :", store.get(namespace, "depot").value)
print("u1 keys :", sorted(item.key for item in store.search(namespace)))
print("u2 :", [item.value for item in store.search(("users", "u2", "profile"))])
store.put(namespace, "depot", {"text": "depot: West"})
print("updated :", store.get(namespace, "depot").value)
store.delete(namespace, "language")
print("deleted :", store.get(namespace, "language"))

profile : {'text': 'depot: Noord'}
u1 keys : ['depot', 'language']
u2 : [{'text': 'depot: Zuid'}]
updated : {'text': 'depot: West'}
deleted : None


We pass a user id through runtime context and keep it independent of thread_id. The graph retrieves the same profile from separate threads for one user.

In [3]:
from dataclasses import dataclass
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_id: str

class MemoryState(TypedDict):
    depot: str

def recall(state: MemoryState, runtime: Runtime[Context]):
    item = runtime.store.get(("users", runtime.context.user_id, "profile"), "depot")
    return {"depot": item.value["text"] if item else "unknown"}

builder = StateGraph(MemoryState, context_schema=Context)
builder.add_node("recall", recall)
builder.add_edge(START, "recall")
builder.add_edge("recall", END)
graph = builder.compile(store=store, checkpointer=InMemorySaver())
for user, thread in [("u1", "u1-morning"), ("u1", "u1-afternoon"), ("u2", "u2-afternoon")]:
    result = graph.invoke({"depot": ""}, {"configurable": {"thread_id": thread}}, context=Context(user))
    print(user, thread, result["depot"])

u1 u1-morning depot: West
u1 u1-afternoon depot: West
u2 u2-afternoon depot: Zuid


We keep ticket facts in a collection and filter by stored fields. Search without an embedding index retrieves matching items; it does not rank by meaning.

In [4]:
import csv

with open(data("demo-world/tickets.csv"), newline="", encoding="utf-8") as file:
    tickets = list(csv.DictReader(file))[:3]
for ticket in tickets:
    store.put(("bus", tickets[0]["fleet_no"], "issues"), ticket["ticket_id"],
              {"text": ticket["symptom"], "error_code": ticket["error_code"]})
hits = store.search(("bus", tickets[0]["fleet_no"], "issues"), filter={"error_code": tickets[0]["error_code"]})
print("filtered :", [(item.key, item.value) for item in hits])

filtered : [('BT-1001', {'text': 'Inverter communication dropout reported by drivers', 'error_code': 'F-CE-410'})]


## Things to try

Optional : we can change a ticket, a thread id or a returned update and compare the saved state.